In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [2]:
import sys, os

sys.path.append(os.environ['LP'])
import project
from project.core.utils import pprint

sys.path.append('../../../param_search')
import param_search as ps

ps.set_verbose(False)
ps.set_backend('slurm')

In [5]:
data_root = '/ocean/projects/asc170022p/mtragoza/lung-project/data/COPDGene'

In [6]:
import project.datasets.copdgene
dataset_cls = project.datasets.copdgene.COPDGeneDataset
ds = dataset_cls(data_root)
ds

project.datasets.copdgene.COPDGeneDataset('/ocean/projects/asc170022p/mtragoza/lung-project/data/COPDGene')

In [7]:
ds.load_metadata()

In [8]:
examples = ds.list_examples(
    subjects='../../data/COPDGene/sample1000_2025-05-21.csv',
    state_pairs=[('EXP', 'INSP')]
)
len(examples)

1000

In [9]:
base_dir = '2026-05-27_preprocess'

template = '''\
#!/bin/bash -l
#SBATCH --job-name={job_name}
#SBATCH --account=asc170022p
#SBATCH --partition=GPU-shared
#SBATCH --gres=gpu:1
#SBATCH -t 6:00:00
set -eo pipefail

LP=$PROJECT/lung-project
NB=$PROJECT/lung-project/notebooks/copdgene

mamba activate /ocean/projects/asc170022p/mtragoza/mambaforge/envs/warp

ln -s network_weights/gradicon_lung1.0/Step_2_final.trch

python $LP/scripts/preprocess.py {config} \\
    --set dataset.name={data_name} \\
    --set dataset.root={data_root} \\
    --set dataset.examples.subjects={subject} \\
    --set dataset.examples.variant={variant} \\

'''
name_format = '{params_hash}'

grid = ps.param_grid(
    config='2026-05-27_config.yaml',
    data_name='COPDGene',
    data_root=data_root,
    subject=[ex.subject for ex in examples],
    variant='2026-05-27'
)
len(grid)

1000

In [10]:
%autoreload
try:
    jobs = ps.setup(base_dir, template, name_format, grid, overwrite=False)
except OSError:
    jobs = ps.load(base_dir)

jobs

,job_name,job_state,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,...,params.subject,params.variant,array_idx,last_live_at,state_source,finalized,finalized_at,output_exists,output_fsize,output_mtime
0,3a1ddc52cf1bce93,COMPLETED,1,41049497,v005,00:01:36,| ├── 'relative_loss': True\n | └──...,,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,16514P,2026-05-27,NaN,None,history,True,2026-05-28T16:01:54,False,None,None
1,7eee1563373873d7,COMPLETED,1,41049498,v029,00:10:29,Reindexing cell labels\nRemoving background ce...,\nRunning sliver perturbation...\nLegend of th...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,20748Q,2026-05-27,NaN,None,history,True,2026-05-28T16:01:54,False,None,None
2,5f1ffa673968c6dd,COMPLETED,1,41049499,v031,00:09:49,Reindexing cell labels\nRemoving background ce...,Total optimization time: 24.3364s\n\nRunning s...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,11007Z,2026-05-27,NaN,None,history,True,2026-05-28T16:01:54,False,None,None
3,0a7e3c5324f4ac8b,COMPLETED,1,41049500,v005,00:07:29,Reindexing cell labels\nRemoving background ce...,\nRunning sliver perturbation...\nLegend of th...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,14771Z,2026-05-27,NaN,None,history,True,2026-05-28T16:01:54,False,None,None
4,b99621050312ac4e,COMPLETED,1,41049501,v029,00:04:28,Reindexing cell labels\nRemoving background ce...,\nRunning sliver perturbation...\nLegend of th...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,13651K,2026-05-27,NaN,None,history,True,2026-05-28T16:01:54,False,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,e8676ea45de24d49,COMPLETED,2,41101544,v015,00:03:58,Reindexing cell labels\nRemoving background ce...,\nRunning sliver perturbation...\nLegend of th...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,20519B,2026-05-27,NaN,2026-06-01T15:14:28,history,True,2026-06-02T09:45:30,False,None,None
996,3db2c84d0afc5372,COMPLETED,2,41101545,v013,00:03:46,Reindexing cell labels\nRemoving background ce...,\nRunning sliver perturbation...\nLegend of th...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,12294H,2026-05-27,NaN,2026-06-01T15:14:28,history,True,2026-06-02T09:45:30,False,None,None
997,c4a93ce51e790114,COMPLETED,2,41101546,v004,00:03:48,Reindexing cell labels\nRemoving background ce...,Total optimization time: 24.3847s\n\nRunning s...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,23123R,2026-05-27,NaN,2026-06-01T15:14:28,history,True,2026-06-02T09:45:30,False,None,None
998,d50289a79d0250f0,COMPLETED,2,41101547,v013,00:03:28,Reindexing cell labels\nRemoving background ce...,\nRunning sliver perturbation...\nLegend of th...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,16546C,2026-05-27,NaN,2026-06-01T15:14:28,history,True,2026-06-02T09:45:30,False,None,None


In [11]:
%autoreload
jobs = ps.recover(jobs)
jobs = ps.status(jobs)
jobs = ps.history(jobs)

jobs.groupby(['job_state']).count()

,job_name,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,script_path,...,params.subject,params.variant,array_idx,last_live_at,state_source,finalized,finalized_at,output_exists,output_fsize,output_mtime
job_state,,,,,,,,,,,,,,,,,,,,,
COMPLETED,998,998,998,998,998,998,998,998,998,998,...,998,998,0,973,998,998,998,998,0,0
TIMEOUT,2,2,2,2,2,2,2,2,2,2,...,2,2,0,2,2,2,2,2,0,0


In [12]:
jobs = ps.collect(jobs)

In [13]:
query_jobs = jobs[jobs.job_state == 'TIMEOUT']
query_jobs

,job_name,job_state,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,...,params.subject,params.variant,array_idx,last_live_at,state_source,finalized,finalized_at,output_exists,output_fsize,output_mtime
445,f26931de6ded612f,TIMEOUT,3,41113137,v021,12:00:16,├── 'optimizer': dict(len=2)\n | ...,construct initial points (nb_points: 12)\nslur...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,23686J,2026-05-27,NaN,2026-06-01T15:14:28,history,True,2026-06-10T08:45:22,False,<NA>,<NA>
873,c0287e139e9a855e,TIMEOUT,3,41113138,v021,12:00:16,├── 'optimizer': dict(len=2)\n | ...,construct initial points (nb_points: 12)\nslur...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,24634V,2026-05-27,NaN,2026-06-01T15:14:28,history,True,2026-06-10T08:45:22,False,<NA>,<NA>


In [14]:
query_jobs.iloc[0]

job_name                                             f26931de6ded612f
job_state                                                     TIMEOUT
n_submits                                                           3
job_id                                                       41113137
node_id                                                          v021
runtime                                                      12:00:16
stdout                  ├── 'optimizer':       dict(len=2)\n    | ...
stderr              construct initial points (nb_points: 12)\nslur...
base_dir            /ocean/projects/asc170022p/mtragoza/lung-proje...
work_dir            /ocean/projects/asc170022p/mtragoza/lung-proje...
script_path         /ocean/projects/asc170022p/mtragoza/lung-proje...
output_path         /ocean/projects/asc170022p/mtragoza/lung-proje...
log_dir             /ocean/projects/asc170022p/mtragoza/lung-proje...
stdout_path         /ocean/projects/asc170022p/mtragoza/lung-proje...
stderr_path         

In [15]:
print(query_jobs.iloc[0].stderr)

construct initial points (nb_points: 12)
slurmstepd: error: *** JOB 41113137 ON v021 CANCELLED AT 2026-06-02T22:04:56 DUE TO TIME LIMIT ***



In [15]:
import pandas as pd
jobs.loc[jobs.job_state == 'TIMEOUT', 'job_id'] = pd.NA

In [18]:
jobs = ps.submit(jobs, time='12:00:00')
jobs

,job_name,job_state,n_submits,job_id,node_id,runtime,stdout,stderr,base_dir,work_dir,...,params.subject,params.variant,array_idx,last_live_at,state_source,finalized,finalized_at,output_exists,output_fsize,output_mtime
0,3a1ddc52cf1bce93,COMPLETED,1,41049497,v005,00:01:36,| ├── 'relative_loss': True\n | └──...,,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,16514P,2026-05-27,NaN,None,history,True,2026-05-28T16:01:54,False,<NA>,<NA>
1,7eee1563373873d7,COMPLETED,1,41049498,v029,00:10:29,Reindexing cell labels\nRemoving background ce...,\nRunning sliver perturbation...\nLegend of th...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,20748Q,2026-05-27,NaN,None,history,True,2026-05-28T16:01:54,False,<NA>,<NA>
2,5f1ffa673968c6dd,COMPLETED,1,41049499,v031,00:09:49,Reindexing cell labels\nRemoving background ce...,Total optimization time: 24.3364s\n\nRunning s...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,11007Z,2026-05-27,NaN,None,history,True,2026-05-28T16:01:54,False,<NA>,<NA>
3,0a7e3c5324f4ac8b,COMPLETED,1,41049500,v005,00:07:29,Reindexing cell labels\nRemoving background ce...,\nRunning sliver perturbation...\nLegend of th...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,14771Z,2026-05-27,NaN,None,history,True,2026-05-28T16:01:54,False,<NA>,<NA>
4,b99621050312ac4e,COMPLETED,1,41049501,v029,00:04:28,Reindexing cell labels\nRemoving background ce...,\nRunning sliver perturbation...\nLegend of th...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,13651K,2026-05-27,NaN,None,history,True,2026-05-28T16:01:54,False,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,e8676ea45de24d49,COMPLETED,2,41101544,v015,00:03:58,Reindexing cell labels\nRemoving background ce...,\nRunning sliver perturbation...\nLegend of th...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,20519B,2026-05-27,NaN,2026-06-01T15:14:28,history,True,2026-06-02T09:45:30,False,<NA>,<NA>
996,3db2c84d0afc5372,COMPLETED,2,41101545,v013,00:03:46,Reindexing cell labels\nRemoving background ce...,\nRunning sliver perturbation...\nLegend of th...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,12294H,2026-05-27,NaN,2026-06-01T15:14:28,history,True,2026-06-02T09:45:30,False,<NA>,<NA>
997,c4a93ce51e790114,COMPLETED,2,41101546,v004,00:03:48,Reindexing cell labels\nRemoving background ce...,Total optimization time: 24.3847s\n\nRunning s...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,23123R,2026-05-27,NaN,2026-06-01T15:14:28,history,True,2026-06-02T09:45:30,False,<NA>,<NA>
998,d50289a79d0250f0,COMPLETED,2,41101547,v013,00:03:28,Reindexing cell labels\nRemoving background ce...,\nRunning sliver perturbation...\nLegend of th...,/ocean/projects/asc170022p/mtragoza/lung-proje...,/ocean/projects/asc170022p/mtragoza/lung-proje...,...,16546C,2026-05-27,NaN,2026-06-01T15:14:28,history,True,2026-06-02T09:45:30,False,<NA>,<NA>
